# Does the amount of Netflix subscribers affect the amount of new content added per quarter?

**Importing and basic steps**

In [1]:
import pandas as pd
import seaborn as sns
import numpy as np
import datetime as dt
import re

import matplotlib.pyplot as plt
%matplotlib inline
from matplotlib.pylab import rcParams
rcParams['figure.figsize'] = 16, 8

from scipy.stats import norm
from scipy import stats

import warnings
warnings.filterwarnings('ignore')
pd.set_option("display.max_columns", None)

pd.options.display.max_rows = 10
np.set_printoptions(precision=4, suppress=True)

## **Data acquisition**

Certain data is ignored from these data sets, but that will become apparent later. Additionally, we got "movies" from https://www.kaggle.com/shivamb/netflix-shows , "subs2021" from https://www.comparitech.com/tv-streaming/netflix-subscribers/ , and "prev_subs" from https://www.kaggle.com/pariaagharabi/netflix2020?select=DataNetflixSubscriber2020_V2.csv

In [2]:
movies = pd.read_csv('netflix_titles.csv')
prev_subs = pd.read_csv('DataNetflixSubscriber2020_V2.csv')
subs2021 = pd.read_csv('subscribers-2021.csv')

In [3]:
movies

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...
...,...,...,...,...,...,...,...,...,...,...,...,...
8802,s8803,Movie,Zodiac,David Fincher,"Mark Ruffalo, Jake Gyllenhaal, Robert Downey J...",United States,"November 20, 2019",2007,R,158 min,"Cult Movies, Dramas, Thrillers","A political cartoonist, a crime reporter and a..."
8803,s8804,TV Show,Zombie Dumb,NaN,NaN,NaN,"July 1, 2019",2018,TV-Y7,2 Seasons,"Kids' TV, Korean TV Shows, TV Comedies","While living alone in a spooky town, a young g..."
8804,s8805,Movie,Zombieland,Ruben Fleischer,"Jesse Eisenberg, Woody Harrelson, Emma Stone, ...",United States,"November 1, 2019",2009,R,88 min,"Comedies, Horror Movies",Looking to survive in a world taken over by zo...
8805,s8806,Movie,Zoom,Peter Hewitt,"Tim Allen, Courteney Cox, Chevy Chase, Kate Ma...",United States,"January 11, 2020",2006,PG,88 min,"Children & Family Movies, Comedies","Dragged from civilian life, a former superhero..."


In [4]:
prev_subs

,Area,Years,Subscribers
0,United States and Canada,Q1 - 2018,60909000
1,"Europe, Middle East and Africa",Q1 - 2018,29339000
2,Latin America,Q1 - 2018,21260000
3,Asia-Pacific,Q1 - 2018,7394000
4,United States and Canada,Q2 - 2018,61870000
...,...,...,...
35,Asia-Pacific,Q1 - 2020,19835000
36,United States and Canada,Q2 - 2020,72904000
37,"Europe, Middle East and Africa",Q2 - 2020,61483000
38,Latin America,Q2 - 2020,36068000


In [5]:
subs2021

,Country,# of Subscribers Q1 2021,Average Monthly Revenue per Paying Membership - Q1 2021 ($),Q1 2021 Revenue $,# of Subscribers Q2 2021,Average Monthly Revenue per Paying Membership - Q2 2021 ($),Q2 2021 Revenue $,# of Subscribers Q3 2021 (Estimate),Q3 2021 Revenue $ (Estimate),# of Subscribers Q4 2021 (Estimate),Q4 2021 Revenue $ (Estimate)
0,Argentina,4968423,7.39,110149942,5069282,7.50,114058849,5154101,115967282,5240340,117907646
1,Australia,6169026,9.71,179703740,6405917,9.74,187180896,6513101,190312806,6622078,193497120
2,Austria,927420,11.56,32162926,930017,11.66,32531987,945578,33076312,961399,33629745
3,Belgium,1639040,11.56,56841907,1643629,11.66,57494153,1671131,58456146,1699092,59434234
4,Brazil,17858800,7.39,395929594,18221334,7.50,409980005,18526213,416839790,18836193,423814354
...,...,...,...,...,...,...,...,...,...,...,...
45,Turkey,3081300,11.56,106859484,3089928,11.66,108085669,3141628,109894158,3194194,111732907
46,UAE,425993,11.56,14773428,427186,11.66,14942949,434333,15192975,441600,15447183
47,United Kingdom,12716400,11.56,441004752,12752006,11.66,446065167,12965372,453528729,13182309,461117172
48,United States,67670924,14.25,2892932001,67278433,14.54,2934685232,68404135,2983788385,69548673,3033713132


## **movies data cleaning** 

Make sure there are no NAs or duplicates, then extracting the data we will use from this dataset. The shows that have NAs in the date added section are not regarded, as they do not help us in our research.

In [6]:
len(movies) - len(movies.drop_duplicates()) #shows no duplicates

0

In [7]:
date_added = movies['date_added'].dropna()
len(movies) - len(date_added) # shows there were 10 NAs

10

In [8]:
movies['date_added'] = pd.to_datetime(movies['date_added']).dt.strftime('%Y-%m').astype('period[Q]') #changing to datetime

movies_per_q = movies['date_added'].value_counts()
movies_per_q = pd.DataFrame(movies_per_q).reset_index().sort_values(by='index').rename(columns={'index': 'Years'})

## **prev_subs data cleaning**

Make sure there are no NAs, duplicates or abnormalities, then extracting the data we will use from this dataset. Since our next dataset is for the year 2021, and we have no data for Q3 and Q4 2020, we will discuss how to deal with these NAs in a following section.

In [9]:
len(prev_subs) - len(prev_subs.drop_duplicates()) #shows there are no duplicates

0

In [10]:
len(prev_subs) - len(prev_subs.dropna()) # shows there are no NAs

0

In [11]:
prev_subs_zscore = (prev_subs['Subscribers'] - prev_subs['Subscribers'].mean())/prev_subs['Subscribers'].std(ddof=0)
prev_subs_outliers = (abs(prev_subs_zscore)>3).astype(int)
print(sum(prev_subs_outliers==1))

0


In [12]:
prev_subs['Years'] = prev_subs['Years'].str.replace(r'(Q\d) - (\d+)', r'\2-\1') #better format to convert 
prev_subs['Years'] = pd.to_datetime(prev_subs['Years']).dt.to_period('Q') #converting to datetime

prev_subs_quarters = prev_subs[['Years','Subscribers']].groupby(['Years']).sum(min_count=1).reset_index()

## **subs2021 data cleaning**

Make sure there are no NAs or duplicates, then extracting the data we will use from this dataset. From this we can also see one abnormaility in subscribers, when identified, it is the USA.

In [13]:
len(subs2021) - len(subs2021.drop_duplicates()) #shows there are no duplicates

0

In [14]:
len(subs2021) - len(subs2021.dropna()) # shows there are no NAs

0

In [15]:
subs2021_q1_zscore = (subs2021['# of Subscribers Q1 2021'] - subs2021['# of Subscribers Q1 2021'].mean())/subs2021['# of Subscribers Q1 2021'].std(ddof=0)
subs2021_q1_outliers = (abs(subs2021_q1_zscore)>3).astype(int)
print(sum(subs2021_q1_outliers==1))

1


In [16]:
subs2021_q2_zscore = (subs2021['# of Subscribers Q2 2021'] - subs2021['# of Subscribers Q2 2021'].mean())/subs2021['# of Subscribers Q2 2021'].std(ddof=0)
subs2021_q2_outliers = (abs(subs2021_q2_zscore)>3).astype(int)
print(sum(subs2021_q2_outliers==1))

1


In [17]:
subs2021_q3_zscore = (subs2021['# of Subscribers Q3 2021 (Estimate)'] - subs2021['# of Subscribers Q3 2021 (Estimate)'].mean())/subs2021['# of Subscribers Q3 2021 (Estimate)'].std(ddof=0)
subs2021_q3_outliers = (abs(subs2021_q3_zscore)>3).astype(int)
print(sum(subs2021_q3_outliers==1))

1


In [18]:
subs2021_q4_zscore = (subs2021['# of Subscribers Q4 2021 (Estimate)'] - subs2021['# of Subscribers Q4 2021 (Estimate)'].mean())/subs2021['# of Subscribers Q4 2021 (Estimate)'].std(ddof=0)
subs2021_q4_outliers = (abs(subs2021_q4_zscore)>3).astype(int)
print(sum(subs2021_q4_outliers==1))

1


In [19]:
other_subs = [subs2021['# of Subscribers Q1 2021'].sum(), #adding 2021 values
            subs2021['# of Subscribers Q2 2021'].sum(),
            subs2021['# of Subscribers Q3 2021 (Estimate)'].sum()]


## **Merging the three dataframes together**

Here we leave out Q4 of 2021, because the dataset we have for the content added was published before the end of 2021

In [20]:
movies_subs = pd.merge(prev_subs_quarters,movies_per_q,on='Years', how='outer') 
movies_subs = movies_subs.rename(columns={'date_added':'Movies Added'})

In [21]:
movies_subs['Subscribers'][40] = other_subs[0]
movies_subs['Subscribers'][41] = other_subs[1]
movies_subs['Subscribers'][42] = other_subs[2]

In [22]:
date = movies_subs['Years'][37]

movies_subs.drop(movies_subs[movies_subs.Years <= date].index, inplace=True) #getting rid of dates before 2018
movies_subs = movies_subs.reset_index()

## **Dealing with NAs of Q3,4 of 2020**

Using values found on other sources: https://www.statista.com/statistics/250934/quarterly-number-of-netflix-streaming-subscribers-worldwide/

In [23]:
sources_movies_subs = movies_subs.copy()
sources_movies_subs['Subscribers'][10] = 195151000
sources_movies_subs['Subscribers'][11] = 203670000

Using the mean of the other values

In [24]:
mean_movies_subs = movies_subs.fillna(movies_subs['Subscribers'][9:13].mean())

TypeError: value should be a 'Period', 'NaT', or array of those. Got 'float' instead.

Using the last known value

In [ ]:
ffill_movies_subs = movies_subs.fillna(method='ffill')

## Making a list of the differences in subscribers

differences using online sources for NA data

In [ ]:
subs = sources_movies_subs['Subscribers']
subs = list(subs)

n=[]
n.append(0)
for i in range(1,len(subs)):
        n.append(subs[i] - subs[i-1])

sources_movies_subs['New Subs'] = n

differences using the mean of known values for NA data

In [ ]:
subs = mean_movies_subs['Subscribers']
subs = list(subs)

n=[]
n.append(0)
for i in range(1,len(subs)):
        n.append(subs[i] - subs[i-1])

mean_movies_subs['New Subs'] = n

differences using the last known value to fill NA values

In [ ]:
subs = ffill_movies_subs['Subscribers']
subs = list(subs)

n=[]
n.append(0)
for i in range(1,len(subs)):
        n.append(subs[i] - subs[i-1])

ffill_movies_subs['New Subs'] = n

## **Visualization**

Using values found from sources

In [ ]:
sources_fig = plt.figure()

covid_per = pd.Period('2020Q1', freq='Q-JAN')

sources_ax1 = sources_fig.add_subplot(2, 2, 1)
sources_movies_subs.plot(kind='line',style='ro-',x='Years',y='Subscribers', ax=sources_ax1)
sources_ax1.annotate('First COVID lockdown (Wuhan)', xy=(covid_per, sources_movies_subs.set_index('Years')['Subscribers'].asof(covid_per)+75),
                  xytext = (covid_per, sources_movies_subs.set_index('Years')['Subscribers'].asof(covid_per)+225),
                  arrowprops = dict(facecolor='black', headwidth=4, width=2, headlength=4),
                  horizontalalignment='left', verticalalignment='top')
sources_ax1.set_title('Total Netflix Subscribers (sources)')


sources_ax2 = sources_fig.add_subplot(2, 2, 2)
sources_movies_subs.plot(kind='line',style='ro-',x='Years',y='Movies Added', ax=sources_ax2)
sources_ax2.annotate('First COVID\nlockdown\n(Wuhan)', xy=(covid_per, sources_movies_subs.set_index('Years')['Movies Added'].asof(covid_per)),
                  xytext = (covid_per, sources_movies_subs.set_index('Years')['Movies Added'].asof(covid_per)+200),
                  arrowprops = dict(facecolor='black', headwidth=4, width=2, headlength=4),
                  horizontalalignment='left', verticalalignment='top')
sources_ax2.set_title('Total Netflix Content Added (sources)')


sources_ax3 = sources_fig.add_subplot(2, 2, 3)
sources_movies_subs.plot(kind='line', style='ro-', x='Years',y='New Subs', ax=sources_ax3)
sources_ax3.annotate('First COVID lockdown\n(Wuhan)', xy=(covid_per, sources_movies_subs.set_index('Years')['New Subs'].asof(covid_per)+75),
                  xytext = (covid_per, sources_movies_subs.set_index('Years')['New Subs'].asof(covid_per)+225),
                  arrowprops = dict(facecolor='black', headwidth=4, width=2, headlength=4),
                  horizontalalignment='left', verticalalignment='top')
sources_ax3.set_title('New Netflix Subscribers (mean)')

plt.subplots_adjust(wspace=0.1, hspace=0.3)

Using the mean values

In [ ]:
mean_fig = plt.figure()

mean_ax1 = mean_fig.add_subplot(2, 2, 1)
mean_movies_subs.plot(kind='line',style='ro-',x='Years',y='Subscribers', ax=mean_ax1)
mean_ax1.annotate('First COVID lockdown (Wuhan)', xy=(covid_per, mean_movies_subs.set_index('Years')['Subscribers'].asof(covid_per)+75),
                  xytext = (covid_per, mean_movies_subs.set_index('Years')['Subscribers'].asof(covid_per)+225),
                  arrowprops = dict(facecolor='black', headwidth=4, width=2, headlength=4),
                  horizontalalignment='left', verticalalignment='top')
mean_ax1.set_title('Total Netflix Subscribers (mean)')


mean_ax2 = mean_fig.add_subplot(2, 2, 2)
mean_movies_subs.plot(kind='line',style='ro-',x='Years',y='Movies Added', ax=mean_ax2)
mean_ax2.annotate('First COVID\nlockdown\n(Wuhan)', xy=(covid_per, mean_movies_subs.set_index('Years')['Movies Added'].asof(covid_per)),
                  xytext = (covid_per, mean_movies_subs.set_index('Years')['Movies Added'].asof(covid_per)+200),
                  arrowprops = dict(facecolor='black', headwidth=4, width=2, headlength=4),
                  horizontalalignment='left', verticalalignment='top')
mean_ax2.set_title('Total Netflix Content Added (mean)')


mean_ax3 = mean_fig.add_subplot(2, 2, 3)
mean_movies_subs.plot(kind='line', style='ro-', x='Years',y='New Subs', ax=mean_ax3)
mean_ax3.annotate('First COVID lockdown\n(Wuhan)', xy=(covid_per, mean_movies_subs.set_index('Years')['New Subs'].asof(covid_per)+75),
                  xytext = (covid_per, mean_movies_subs.set_index('Years')['New Subs'].asof(covid_per)+225),
                  arrowprops = dict(facecolor='black', headwidth=4, width=2, headlength=4),
                  horizontalalignment='left', verticalalignment='top')
mean_ax3.set_title('New Netflix Subscribers (mean)')

plt.subplots_adjust(wspace=0.1, hspace=0.3)

Using the last known values

In [ ]:
ffill_fig = plt.figure()

ffill_ax1 = ffill_fig.add_subplot(2, 2, 1)
ffill_movies_subs.plot(kind='line',style='ro-',x='Years',y='Subscribers', ax=ffill_ax1)
ffill_ax1.annotate('First COVID lockdown (Wuhan)', xy=(covid_per, ffill_movies_subs.set_index('Years')['Subscribers'].asof(covid_per)+75),
                  xytext = (covid_per, ffill_movies_subs.set_index('Years')['Subscribers'].asof(covid_per)+225),
                  arrowprops = dict(facecolor='black', headwidth=4, width=2, headlength=4),
                  horizontalalignment='left', verticalalignment='top')
ffill_ax1.set_title('Total Netflix Subscribers (ffill)')


ffill_ax2 = ffill_fig.add_subplot(2, 2, 2)
ffill_movies_subs.plot(kind='line',style='ro-',x='Years',y='Movies Added', ax=ffill_ax2)
ffill_ax2.annotate('First COVID\nlockdown\n(Wuhan)', xy=(covid_per, ffill_movies_subs.set_index('Years')['Movies Added'].asof(covid_per)),
                  xytext = (covid_per, ffill_movies_subs.set_index('Years')['Movies Added'].asof(covid_per)+200),
                  arrowprops = dict(facecolor='black', headwidth=4, width=2, headlength=4),
                  horizontalalignment='left', verticalalignment='top')
ffill_ax2.set_title('Total Netflix Content Added (ffill)')


ffill_ax3 = ffill_fig.add_subplot(2, 2, 3)
ffill_movies_subs.plot(kind='line', style='ro-', x='Years',y='New Subs', ax=ffill_ax3)
ffill_ax3.annotate('First COVID lockdown\n(Wuhan)', xy=(covid_per, ffill_movies_subs.set_index('Years')['New Subs'].asof(covid_per)+75),
                  xytext = (covid_per, ffill_movies_subs.set_index('Years')['New Subs'].asof(covid_per)+225),
                  arrowprops = dict(facecolor='black', headwidth=4, width=2, headlength=4),
                  horizontalalignment='left', verticalalignment='top')
ffill_ax3.set_title('New Netflix Subscribers (ffill)')

plt.subplots_adjust(wspace=0.1, hspace=0.3)


In [ ]:
overview_fig = plt.figure()

overview_ax1 = overview_fig.add_subplot(2, 2, 1)
sources_movies_subs.plot(kind='line', style='ro-', x='Years',y='New Subs', ax=overview_ax1)
overview_ax1.set_title('New Netflix Subscribers (sources)')

overview_ax2 = overview_fig.add_subplot(2, 2, 2)
mean_movies_subs.plot(kind='line', style='ro-', x='Years',y='New Subs', ax=overview_ax2)
overview_ax2.set_title('New Netflix Subscribers (mean)')

overview_ax3 = overview_fig.add_subplot(2, 2, 3)
ffill_movies_subs.plot(kind='line', style='ro-', x='Years',y='New Subs', ax=overview_ax3)
overview_ax3.set_title('New Netflix Subscribers (ffill)')

overview_ax4 = overview_fig.add_subplot(2, 2, 4)
ffill_movies_subs.plot(kind='line',style='ro-',x='Years',y='Movies Added', ax=overview_ax4)
overview_ax4.set_title('Total Netflix Content Added (ffill)')

plt.subplots_adjust(wspace=0.1, hspace=0.3)

## Numerical summary

In [ ]:
corr,_ = stats.pearsonr(sources_movies_subs['New Subs'], sources_movies_subs['Movies Added'])
print('Pearsons correlation: %.3f' % corr)
corr,_ = stats.spearmanr(sources_movies_subs['New Subs'], sources_movies_subs['Movies Added'])
print('Spearmans correlation: %.3f' % corr)

np.cov(sources_movies_subs['New Subs'],sources_movies_subs['Movies Added']) 

In [ ]:
corr,_ = stats.pearsonr(mean_movies_subs['New Subs'], mean_movies_subs['Movies Added'])
print('Pearsons correlation: %.3f' % corr)
corr,_ = stats.spearmanr(mean_movies_subs['New Subs'], mean_movies_subs['Movies Added'])
print('Spearmans correlation: %.3f' % corr)

np.cov(mean_movies_subs['New Subs'],mean_movies_subs['Movies Added']) 

In [ ]:
corr,_ = stats.pearsonr(ffill_movies_subs['New Subs'], ffill_movies_subs['Movies Added'])
print('Pearsons correlation: %.3f' % corr)
corr,_ = stats.spearmanr(ffill_movies_subs['New Subs'], ffill_movies_subs['Movies Added'])
print('Spearmans correlation: %.3f' % corr)

np.cov(ffill_movies_subs['New Subs'],ffill_movies_subs['Movies Added']) 